# Week 1 - Russell 1000 Simple Analysis

This notebook is the Week 1 analysis scaffold for a Russell 1000 universe. It works with either WRDS exports or Databento/API exports saved as CSV files.

Expected inputs:
- `data/raw/index/russell1000_membership.csv`
- `data/raw/market/daily_prices.csv`
- optional: `data/raw/events/ma_events.csv`

If inputs are missing, the notebook writes CSV templates and exits cleanly.

In [1]:
from __future__ import annotations

import importlib.util
import json
import os
from pathlib import Path

import pandas as pd


def resolve_project_root() -> Path:
    cwd = Path.cwd().resolve()
    for candidate in [cwd, cwd.parent]:
        if (candidate / "README.md").exists() and (candidate / "code" / "README.md").exists():
            return candidate
    return cwd


PROJECT_ROOT = resolve_project_root()
INDEX_PATH = PROJECT_ROOT / "data" / "raw" / "index" / "us_listed_companies_sec.csv"
PRICE_PATH = PROJECT_ROOT / "data" / "raw" / "market" / "daily_prices.csv"
EVENT_PATH = PROJECT_ROOT / "data" / "raw" / "events" / "ma_events.csv"
REPORT_DIR = PROJECT_ROOT / "reports" / "week1" / "russell1000_analysis"
REPORT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Project root: {PROJECT_ROOT}")
print(f"Membership path: {INDEX_PATH}")
print(f"Price path: {PRICE_PATH}")

Project root: /Users/cindyfu/Desktop/UCB/Industry Project - JPM
Membership path: /Users/cindyfu/Desktop/UCB/Industry Project - JPM/data/raw/index/us_listed_companies_sec.csv
Price path: /Users/cindyfu/Desktop/UCB/Industry Project - JPM/data/raw/market/daily_prices.csv


## 1. Create Input Templates If Needed

In [2]:
templates = {
    INDEX_PATH: ["ticker", "company_name", "sector", "start_date", "end_date", "permno", "gvkey", "cik"],
    PRICE_PATH: ["date", "ticker", "open", "high", "low", "close", "volume", "ret", "shares_out", "market_cap", "permno", "gvkey"],
    EVENT_PATH: ["announcement_date", "target_ticker", "acquirer_ticker", "deal_status", "close_date", "deal_value_usd"],
}

created_templates = []
for path, columns in templates.items():
    path.parent.mkdir(parents=True, exist_ok=True)
    if not path.exists():
        pd.DataFrame(columns=columns).to_csv(path, index=False)
        created_templates.append(str(path.relative_to(PROJECT_ROOT)))

if created_templates:
    print("Created missing templates:")
    for item in created_templates:
        print(f"- {item}")
else:
    print("All expected input files exist.")

All expected input files exist.


## 2. Optional Databento Download

This cell downloads daily OHLCV bars from Databento into `data/raw/market/daily_prices.csv`.

Important:
- Keep the API key out of the notebook. Set `DATABENTO_API_KEY` in your shell or notebook environment.
- By default this cell downloads only tickers appearing in `data/raw/events/ma_events.csv`, so it stays below Databento's per-request symbol limit.
- Switch `DATABENTO_SYMBOL_SOURCE` to `"index"` only if you intentionally want the broader universe.
- Start with a short date range to control cost, then expand once the schema is confirmed.

In [3]:
RUN_DATABENTO_DOWNLOAD = True  # Set to False to skip downloading from Databento. Make sure to set DATABENTO_API_KEY in your environment if True.

# Common Databento US equities choices. Change these if your subscription uses another dataset.
DATABENTO_DATASET = "XNAS.ITCH"
DATABENTO_SCHEMA = "ohlcv-1d"
DATABENTO_STYPE_IN = "raw_symbol"
DATABENTO_START = "2026-01-01"
DATABENTO_END = "2026-03-01"
DATABENTO_SYMBOL_SOURCE = "ma_events"  # Use "ma_events" for event-related companies, or "index" for the full universe.
DATABENTO_SYMBOL_LIMIT = None  # Set to None for no limit, or a positive integer to limit the number of selected symbols.
DATABENTO_MAX_SYMBOLS_PER_REQUEST = 2000


def load_local_env(path: Path = PROJECT_ROOT / "code" / ".env") -> None:
    if not path.exists():
        return
    for raw_line in path.read_text(encoding="utf-8").splitlines():
        line = raw_line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        key, value = line.split("=", 1)
        key = key.strip().strip('\"').strip("'")
        value = value.strip().strip('\"').strip("'")
        os.environ.setdefault(key, value)


load_local_env()


def get_universe_symbols_for_databento(path: Path, limit: int | None = None) -> list[str]:
    if not path.exists():
        return []
    frame = pd.read_csv(path)
    if frame.empty or "ticker" not in frame.columns:
        return []
    symbols = (
        frame["ticker"]
        .dropna()
        .astype(str)
        .str.strip()
        .str.upper()
        .drop_duplicates()
        .tolist()
    )
    return symbols[:limit] if limit else symbols


def get_ma_event_symbols_for_databento(path: Path, limit: int | None = None) -> list[str]:
    if not path.exists():
        return []
    frame = pd.read_csv(path)
    ticker_columns = [col for col in ["target_ticker", "acquirer_ticker"] if col in frame.columns]
    if frame.empty or not ticker_columns:
        return []
    symbols = (
        pd.concat([frame[col] for col in ticker_columns], ignore_index=True)
        .dropna()
        .astype(str)
        .str.strip()
        .str.upper()
    )
    symbols = symbols.loc[~symbols.isin(["", "NAN", "NONE", "<NA>"])].drop_duplicates().tolist()
    return symbols[:limit] if limit else symbols


def get_databento_symbols(limit: int | None = None) -> list[str]:
    if DATABENTO_SYMBOL_SOURCE == "ma_events":
        symbols = get_ma_event_symbols_for_databento(EVENT_PATH, limit)
        source_path = EVENT_PATH
    elif DATABENTO_SYMBOL_SOURCE == "index":
        symbols = get_universe_symbols_for_databento(INDEX_PATH, limit)
        source_path = INDEX_PATH
    else:
        raise ValueError('DATABENTO_SYMBOL_SOURCE must be "ma_events" or "index".')

    if len(symbols) > DATABENTO_MAX_SYMBOLS_PER_REQUEST:
        raise ValueError(
            f"Selected {len(symbols):,} Databento symbols from {source_path.relative_to(PROJECT_ROOT)}, "
            f"which exceeds the {DATABENTO_MAX_SYMBOLS_PER_REQUEST:,}-symbol request limit. "
            "Use DATABENTO_SYMBOL_LIMIT or split the request."
        )
    print(f"Selected {len(symbols):,} Databento symbols from {source_path.relative_to(PROJECT_ROOT)}")
    return symbols


def normalize_databento_ohlcv(frame: pd.DataFrame) -> pd.DataFrame:
    if frame.empty:
        return frame

    out = frame.reset_index()
    rename_map = {
        "ts_event": "date",
        "symbol": "ticker",
        "open": "open",
        "high": "high",
        "low": "low",
        "close": "close",
        "volume": "volume",
    }
    out = out.rename(columns={k: v for k, v in rename_map.items() if k in out.columns})
    if "date" not in out.columns:
        date_like = [col for col in out.columns if "time" in col.lower() or "date" in col.lower()]
        if date_like:
            out = out.rename(columns={date_like[0]: "date"})
    if "ticker" not in out.columns and "instrument_id" in out.columns:
        out["ticker"] = out["instrument_id"].astype(str)

    keep = [col for col in ["date", "ticker", "open", "high", "low", "close", "volume"] if col in out.columns]
    out = out[keep].copy()
    out["date"] = pd.to_datetime(out["date"], errors="coerce").dt.date
    out["ticker"] = out["ticker"].astype(str).str.upper()
    for col in ["open", "high", "low", "close", "volume"]:
        if col in out.columns:
            out[col] = pd.to_numeric(out[col], errors="coerce")
    out = out.dropna(subset=["date", "ticker"]).sort_values(["ticker", "date"])
    out["ret"] = out.groupby("ticker")["close"].pct_change()
    for col in ["shares_out", "market_cap", "permno", "gvkey"]:
        if col not in out.columns:
            out[col] = pd.NA
    return out[["date", "ticker", "open", "high", "low", "close", "volume", "ret", "shares_out", "market_cap", "permno", "gvkey"]]


def download_databento_daily_prices() -> pd.DataFrame:
    if importlib.util.find_spec("databento") is None:
        raise ImportError("Install the Databento Python client first: python3 -m pip install databento")
    api_key = os.getenv("DATABENTO_API_KEY")
    if not api_key:
        raise ValueError("Set DATABENTO_API_KEY in your environment or code/.env before running the download cell.")

    import databento as db

    symbols = get_databento_symbols(DATABENTO_SYMBOL_LIMIT)
    if not symbols:
        raise ValueError("No Databento symbols selected. Populate data/raw/events/ma_events.csv or switch DATABENTO_SYMBOL_SOURCE to 'index'.")

    client = db.Historical(api_key)
    data = client.timeseries.get_range(
        dataset=DATABENTO_DATASET,
        schema=DATABENTO_SCHEMA,
        symbols=symbols,
        stype_in=DATABENTO_STYPE_IN,
        start=DATABENTO_START,
        end=DATABENTO_END,
    )
    frame = normalize_databento_ohlcv(data.to_df())
    PRICE_PATH.parent.mkdir(parents=True, exist_ok=True)
    frame.to_csv(PRICE_PATH, index=False)
    return frame


databento_download_status = {"attempted": RUN_DATABENTO_DOWNLOAD, "status": "skipped", "rows": 0, "error": ""}
if RUN_DATABENTO_DOWNLOAD:
    try:
        downloaded_prices = download_databento_daily_prices()
        databento_download_status.update({"status": "ok", "rows": len(downloaded_prices)})
        print(f"Downloaded {len(downloaded_prices):,} Databento rows to {PRICE_PATH.relative_to(PROJECT_ROOT)}")
    except Exception as exc:
        databento_download_status.update({"status": "failed", "error": repr(exc)})
        print(f"Databento download failed; continuing with existing local CSV if available. Error: {exc!r}")
else:
    print("Databento download skipped. Set RUN_DATABENTO_DOWNLOAD = True after setting DATABENTO_API_KEY.")

Selected 235 Databento symbols from data/raw/events/ma_events.csv


/var/folders/yp/_v837mn16kz84p2t4pb166xc0000gn/T/ipykernel_7082/3581958537.py:137: BentoWarning: The streaming request had one or more symbols which did not resolve: IPFX, GRML, QRED...
  data = client.timeseries.get_range(


Downloaded 8,445 Databento rows to data/raw/market/daily_prices.csv


## 3. Load Universe and Price Data

In [4]:
def read_csv_if_populated(path: Path, date_columns: list[str] | None = None) -> pd.DataFrame:
    frame = pd.read_csv(path)
    if frame.empty:
        return frame
    for col in date_columns or []:
        if col in frame.columns:
            frame[col] = pd.to_datetime(frame[col], errors="coerce")
    return frame


universe = read_csv_if_populated(INDEX_PATH, ["start_date", "end_date"])
prices = read_csv_if_populated(PRICE_PATH, ["date"])
events = read_csv_if_populated(EVENT_PATH, ["announcement_date", "close_date"])

print(f"Universe rows: {len(universe):,}")
print(f"Price rows: {len(prices):,}")
print(f"M&A event rows: {len(events):,}")

Universe rows: 7,615
Price rows: 8,445
M&A event rows: 200


## 4. Standardize Columns

In [5]:
def standardize_ticker(frame: pd.DataFrame, column: str = "ticker") -> pd.DataFrame:
    if column in frame.columns:
        frame[column] = frame[column].astype(str).str.strip().str.upper()
        frame.loc[frame[column].isin(["", "NAN", "NONE"]), column] = pd.NA
    return frame


universe = standardize_ticker(universe, "ticker")
prices = standardize_ticker(prices, "ticker")
events = standardize_ticker(events, "target_ticker")
events = standardize_ticker(events, "acquirer_ticker")

if not prices.empty:
    numeric_cols = ["open", "high", "low", "close", "volume", "ret", "shares_out", "market_cap"]
    for col in numeric_cols:
        if col in prices.columns:
            prices[col] = pd.to_numeric(prices[col], errors="coerce")
    prices = prices.dropna(subset=["date", "ticker"]).sort_values(["ticker", "date"])
    if "ret" not in prices.columns or prices["ret"].isna().all():
        prices["ret"] = prices.groupby("ticker")["close"].pct_change()
    if "market_cap" not in prices.columns and {"close", "shares_out"}.issubset(prices.columns):
        prices["market_cap"] = prices["close"].abs() * prices["shares_out"]
    prices["dollar_volume"] = prices["close"].abs() * prices["volume"]

universe.head()

,ticker,company_name,exchange,cik,sec_rank,source,source_url
0,A,"AGILENT TECHNOLOGIES, INC.",NYSE,1090872,424,SEC company_tickers_exchange.json,https://www.sec.gov/files/company_tickers_exch...
1,AA,Alcoa Corp,NYSE,1675149,681,SEC company_tickers_exchange.json,https://www.sec.gov/files/company_tickers_exch...
2,AACB,Artius II Acquisition Inc.,Nasdaq,2034334,3929,SEC company_tickers_exchange.json,https://www.sec.gov/files/company_tickers_exch...
3,AACBR,Artius II Acquisition Inc.,Nasdaq,2034334,9853,SEC company_tickers_exchange.json,https://www.sec.gov/files/company_tickers_exch...
4,AACBU,Artius II Acquisition Inc.,Nasdaq,2034334,9854,SEC company_tickers_exchange.json,https://www.sec.gov/files/company_tickers_exch...


## 5. Universe Coverage Summary

In [6]:
if universe.empty:
    universe_summary = pd.DataFrame([{"status": "missing_or_empty", "message": "Populate data/raw/index/russell1000_membership.csv"}])
else:
    universe_summary = pd.DataFrame(
        [
            {"metric": "rows", "value": len(universe)},
            {"metric": "unique_tickers", "value": universe["ticker"].nunique()},
            {"metric": "missing_sector", "value": int(universe.get("sector", pd.Series(dtype=object)).isna().sum())},
            {"metric": "missing_permno", "value": int(universe.get("permno", pd.Series(dtype=object)).isna().sum())},
            {"metric": "missing_gvkey", "value": int(universe.get("gvkey", pd.Series(dtype=object)).isna().sum())},
        ]
    )

universe_summary.to_csv(REPORT_DIR / "universe_summary.csv", index=False)
universe_summary

,metric,value
0,rows,7615
1,unique_tickers,7613
2,missing_sector,0
3,missing_permno,0
4,missing_gvkey,0


In [7]:
if not universe.empty and "sector" in universe.columns:
    sector_distribution = (
        universe.assign(sector=universe["sector"].fillna("Unknown"))
        .groupby("sector", dropna=False)
        .agg(company_count=("ticker", "nunique"))
        .sort_values("company_count", ascending=False)
        .reset_index()
    )
else:
    sector_distribution = pd.DataFrame(columns=["sector", "company_count"])

sector_distribution.to_csv(REPORT_DIR / "sector_distribution.csv", index=False)
sector_distribution.head(20)

,sector,company_count


## 6. Price Coverage and Simple Market Features

In [8]:
if prices.empty:
    price_coverage = pd.DataFrame([{"status": "missing_or_empty", "message": "Populate data/raw/market/daily_prices.csv"}])
    market_feature_summary = pd.DataFrame()
else:
    price_coverage = (
        prices.groupby("ticker")
        .agg(
            first_date=("date", "min"),
            last_date=("date", "max"),
            trading_days=("date", "nunique"),
            missing_close=("close", lambda s: int(s.isna().sum())),
            missing_volume=("volume", lambda s: int(s.isna().sum())),
            avg_dollar_volume=("dollar_volume", "mean"),
            annualized_vol=("ret", lambda s: s.std(skipna=True) * (252 ** 0.5)),
            cumulative_return=("ret", lambda s: (1 + s.dropna()).prod() - 1 if s.notna().any() else pd.NA),
        )
        .reset_index()
    )
    if not universe.empty:
        price_coverage = universe[["ticker"]].drop_duplicates().merge(price_coverage, how="left", on="ticker")
        price_coverage["has_price_data"] = price_coverage["trading_days"].notna()

    market_feature_summary = price_coverage.describe(include="all").reset_index()

price_coverage.to_csv(REPORT_DIR / "price_coverage_by_ticker.csv", index=False)
market_feature_summary.to_csv(REPORT_DIR / "market_feature_summary.csv", index=False)
price_coverage.head(20)

,ticker,first_date,last_date,trading_days,missing_close,missing_volume,avg_dollar_volume,annualized_vol,cumulative_return,has_price_data
0,A,NaT,NaT,NaN,NaN,NaN,NaN,NaN,NaN,False
1,AA,NaT,NaT,NaN,NaN,NaN,NaN,NaN,NaN,False
2,AACB,NaT,NaT,NaN,NaN,NaN,NaN,NaN,NaN,False
3,AACBR,NaT,NaT,NaN,NaN,NaN,NaN,NaN,NaN,False
4,AACBU,NaT,NaT,NaN,NaN,NaN,NaN,NaN,NaN,False
5,AACG,NaT,NaT,NaN,NaN,NaN,NaN,NaN,NaN,False
6,AACI,NaT,NaT,NaN,NaN,NaN,NaN,NaN,NaN,False
7,AACIU,NaT,NaT,NaN,NaN,NaN,NaN,NaN,NaN,False
8,AACIW,NaT,NaT,NaN,NaN,NaN,NaN,NaN,NaN,False
9,AACO,NaT,NaT,NaN,NaN,NaN,NaN,NaN,NaN,False


## 7. Optional M&A Event Summary

In [9]:
if events.empty:
    event_summary = pd.DataFrame([{"status": "missing_or_empty", "message": "Optional: populate data/raw/events/ma_events.csv"}])
else:
    event_summary = pd.DataFrame(
        [
            {"metric": "events", "value": len(events)},
            {"metric": "unique_targets", "value": events["target_ticker"].nunique()},
            {"metric": "unique_acquirers", "value": events["acquirer_ticker"].nunique() if "acquirer_ticker" in events else 0},
            {"metric": "first_announcement", "value": events["announcement_date"].min()},
            {"metric": "last_announcement", "value": events["announcement_date"].max()},
        ]
    )

event_summary.to_csv(REPORT_DIR / "event_summary.csv", index=False)
event_summary

,metric,value
0,events,200
1,unique_targets,195
2,unique_acquirers,43
3,first_announcement,2025-06-20 00:00:00
4,last_announcement,2026-06-09 00:00:00


## 8. Connect M&A Events to the README Prediction Targets

The README defines three M&A tasks: target identification, acquirer identification, and compatible acquisition pairs. This section turns the optional event file into target/acquirer features and 6-month forward labels.

In [10]:
def build_company_ma_features(universe: pd.DataFrame, events: pd.DataFrame) -> pd.DataFrame:
    base_cols = [col for col in ["ticker", "company_name", "sector"] if col in universe.columns]
    base = universe[base_cols].drop_duplicates("ticker").copy() if not universe.empty else pd.DataFrame(columns=["ticker"])
    if events.empty or base.empty:
        return base.assign(
            target_event_count=0,
            acquirer_event_count=0,
            first_target_announcement=pd.NaT,
            first_acquirer_announcement=pd.NaT,
        )

    target_summary = (
        events.dropna(subset=["target_ticker"])
        .groupby("target_ticker")
        .agg(
            target_event_count=("announcement_date", "count"),
            first_target_announcement=("announcement_date", "min"),
        )
        .rename_axis("ticker")
        .reset_index()
    )
    acquirer_summary = (
        events.dropna(subset=["acquirer_ticker"])
        .groupby("acquirer_ticker")
        .agg(
            acquirer_event_count=("announcement_date", "count"),
            first_acquirer_announcement=("announcement_date", "min"),
        )
        .rename_axis("ticker")
        .reset_index()
    )
    out = base.merge(target_summary, how="left", on="ticker").merge(acquirer_summary, how="left", on="ticker")
    out["target_event_count"] = out["target_event_count"].fillna(0).astype(int)
    out["acquirer_event_count"] = out["acquirer_event_count"].fillna(0).astype(int)
    return out


def build_forward_ma_labels(universe: pd.DataFrame, events: pd.DataFrame, prices: pd.DataFrame, horizon_months: int = 6) -> pd.DataFrame:
    if universe.empty or "ticker" not in universe.columns:
        return pd.DataFrame(columns=["ticker", "as_of_date", "label_target_within_6m", "label_acquirer_within_6m"])

    if not prices.empty and "date" in prices.columns:
        as_of_dates = (
            pd.Series(pd.to_datetime(prices["date"], errors="coerce").dropna().unique())
            .sort_values()
            .dt.to_period("M")
            .dt.to_timestamp("M")
            .drop_duplicates()
        )
    else:
        as_of_dates = pd.Series(pd.date_range("2018-01-31", "2024-12-31", freq="ME"))

    tickers = universe[["ticker"]].drop_duplicates().dropna().copy()
    panel = tickers.merge(pd.DataFrame({"as_of_date": as_of_dates}), how="cross")
    panel["horizon_end"] = panel["as_of_date"] + pd.DateOffset(months=horizon_months)

    for role, event_col in [("target", "target_ticker"), ("acquirer", "acquirer_ticker")]:
        label_col = f"label_{role}_within_6m"
        next_col = f"next_{role}_announcement_date"
        if events.empty or event_col not in events.columns:
            panel[label_col] = 0
            panel[next_col] = pd.NaT
            continue

        role_events = (
            events[[event_col, "announcement_date"]]
            .dropna(subset=[event_col, "announcement_date"])
            .rename(columns={event_col: "ticker"})
        )
        merged = panel[["ticker", "as_of_date", "horizon_end"]].merge(role_events, how="left", on="ticker")
        is_forward_event = (merged["announcement_date"] > merged["as_of_date"]) & (merged["announcement_date"] <= merged["horizon_end"])
        labels = (
            merged.loc[is_forward_event]
            .groupby(["ticker", "as_of_date"], as_index=False)
            .agg(**{next_col: ("announcement_date", "min")})
        )
        labels[label_col] = 1
        panel = panel.merge(labels, how="left", on=["ticker", "as_of_date"])
        panel[label_col] = panel[label_col].fillna(0).astype(int)

    return panel.drop(columns=["horizon_end"])


company_ma_features = build_company_ma_features(universe, events)
ma_forward_labels = build_forward_ma_labels(universe, events, prices, horizon_months=6)

company_ma_features.to_csv(REPORT_DIR / "company_ma_features.csv", index=False)
ma_forward_labels.to_csv(REPORT_DIR / "ma_forward_labels_6m.csv", index=False)

print(f"Company M&A feature rows: {len(company_ma_features):,}")
print(f"Forward label rows: {len(ma_forward_labels):,}")
company_ma_features.head()

Company M&A feature rows: 7,614
Forward label rows: 15,226


,ticker,company_name,target_event_count,first_target_announcement,acquirer_event_count,first_acquirer_announcement
0,A,"AGILENT TECHNOLOGIES, INC.",0,NaT,0,NaT
1,AA,Alcoa Corp,0,NaT,0,NaT
2,AACB,Artius II Acquisition Inc.,0,NaT,0,NaT
3,AACBR,Artius II Acquisition Inc.,0,NaT,0,NaT
4,AACBU,Artius II Acquisition Inc.,0,NaT,0,NaT


## 9. Analysis Manifest

In [11]:
manifest = {
    "inputs": {
        "membership": str(INDEX_PATH.relative_to(PROJECT_ROOT)),
        "prices": str(PRICE_PATH.relative_to(PROJECT_ROOT)),
        "events_optional": str(EVENT_PATH.relative_to(PROJECT_ROOT)),
    },
    "outputs": sorted(str(path.relative_to(PROJECT_ROOT)) for path in REPORT_DIR.glob("*.csv")),
    "universe_rows": int(len(universe)),
    "price_rows": int(len(prices)),
    "event_rows": int(len(events)),
}

(REPORT_DIR / "analysis_manifest.json").write_text(json.dumps(manifest, indent=2), encoding="utf-8")
manifest

{'inputs': {'membership': 'data/raw/index/us_listed_companies_sec.csv',
  'prices': 'data/raw/market/daily_prices.csv',
  'events_optional': 'data/raw/events/ma_events.csv'},
 'outputs': ['reports/week1/russell1000_analysis/company_ma_features.csv',
  'reports/week1/russell1000_analysis/event_summary.csv',
  'reports/week1/russell1000_analysis/ma_forward_labels_6m.csv',
  'reports/week1/russell1000_analysis/market_feature_summary.csv',
  'reports/week1/russell1000_analysis/price_coverage_by_ticker.csv',
  'reports/week1/russell1000_analysis/sector_distribution.csv',
  'reports/week1/russell1000_analysis/universe_summary.csv'],
 'universe_rows': 7615,
 'price_rows': 8445,
 'event_rows': 200}